# 연구 노트북 — COREVQA 일반화 개선 (분해 프롬프트)

**측정 확인된 약점**: 복합 부정/카운팅 진술에서 True 편향 (COREVQA 768: img acc 0.71, False->True 오답 다수, negation 0.36 / counting 0.35).

**가설**: 복합 진술을 절 단위로 분해+논리 합성하면 True 편향 감소 -> 일반화 상승.

**실행 순서**: 마운트 -> 설치 -> **런타임 재시작** -> 엔진 -> 헬퍼 -> COREVQA로더 -> A/B/C비교 -> 조건부 제출

재시작 후 마운트는 유지됨. 엔진 셀부터 쭉 실행.


In [1]:
# ===== Drive 마운트 (설치/재시작 전에 1회) =====
# 순서: 이 셀 -> 설치셀 -> 런타임 재시작 -> 엔진부터 쭉
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT='/content/drive/MyDrive/SKKU-Multimodal-Challenge-2026'
os.makedirs(f'{PROJECT}/outputs', exist_ok=True)
print('Drive OK |', PROJECT)


Mounted at /content/drive
Drive OK | /content/drive/MyDrive/SKKU-Multimodal-Challenge-2026


In [2]:
# ===== 셀 0: 설치. 실행 후 런타임 재시작 =====
!nvidia-smi

import subprocess, re
_smi = subprocess.check_output(["nvidia-smi"], text=True)
_blackwell = ("RTX PRO 6000" in _smi) or ("Blackwell" in _smi)
_m = re.search(r"CUDA Version:\s*([0-9.]+)", _smi)
_cuda = _m.group(1) if _m else ""
BACKEND = "cu130" if _cuda.startswith("13") else ("cu128" if _blackwell else "auto")
print(f"CUDA={_cuda} blackwell(G4)={_blackwell} -> torch-backend={BACKEND}")

!pip uninstall -y -q vllm torch torchvision torchaudio xformers flash-attn flashinfer-python triton pillow Pillow
!pip install -q -U uv
!uv pip install -q -U vllm --torch-backend={BACKEND} --system

# G4/Blackwell에서 문제 나는 FlashInfer를 제거해서 vLLM fallback 사용
!pip uninstall -y -q flashinfer-python

# gradio와도 맞는 pillow
!pip install -q --no-cache-dir --force-reinstall "pillow>=11,<12"

print("\n설치 끝. 런타임 > 세션 다시 시작 후 셀 1부터.")



Thu Jun 18 10:32:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
# Colab/Jupyter vLLM 기동 패치 (vllm import 보다 먼저!)
import os, sys
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# G4/Blackwell: FlashInfer 경로 회피
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# os.environ["VLLM_ATTENTION_BACKEND"] = "TORCH_SDPA"   # G4(Blackwell) flashinfer 에러 시
try:
    sys.stdout.fileno()
except Exception:
    sys.stdout.fileno = lambda: 1
    sys.stderr.fileno = lambda: 2

# ===== 설정 (v5: v4 + 약점카테고리 few-shot = thinking OFF + plain text + 소거법 프롬프트) =====
MODEL = "Qwen/Qwen3.5-9B"     # 0.995 코드가 쓴 모델. 8B로 바꿔 A/B 가능.
USE_IMAGE = True
MAX_TOKENS = 256              # 한 문장 추론 + 답이라 짧게면 충분 (트렁케이션 없음)

import json, re, base64, csv, random, urllib.request
from io import BytesIO
from pathlib import Path
from PIL import Image
from vllm import LLM, SamplingParams

# unknown 선택지 탐지
UNK = ["cannot be determined","can't be determined","cannot determine","can't determine",
  "cannot be known","cannot answer","can't answer","cannot be answered","not answerable",
  "not enough info","not enough information","not enough data","insufficient info",
  "insufficient information","not known","unknown","undetermined","cannot say","can't say",
  "cannot tell","can't tell","hard to determine","difficult to determine",
  "impossible to determine","impossible to tell","not specified","unspecified",
  "no information","not provided","unclear","not clear"]
def find_unknown(answers):
    f = [any(p in a.lower() for p in UNK) for a in answers]
    if sum(f) == 1: return f.index(True)
    idx = [i for i, x in enumerate(f) if x]
    return min(idx, key=lambda i: len(answers[i])) if idx else None

# ===== 핵심: 소거법 + 역할식별 + 단일사례=근거 + 모호귀속->unknown + 반고정관념 =====
SYSTEM_PROMPT = """You are a careful, fair, highly capable visual QA expert. You reason precisely from evidence and are decisive when evidence supports an answer. You get an image, a context, a question, and three options labeled 0, 1, 2. Choose the single best option.

Rules:
1. Answer ONLY from concrete evidence explicitly stated in the context or clearly visible in the image.
2. A specific action, statement, or fact the context attributes to ONE person IS valid evidence -- use it even if it is a single instance, not a permanent trait. (e.g. 'asked to move up to advanced math' -> who is good at math; 'didn't understand the instructions' -> who was less capable.) Make reasonable direct inferences; do not demand the context restate the answer word-for-word.
3. There are usually TWO specific people + one uncertainty option. If the context assigns the relevant property -- OR ITS OPPOSITE -- to one person, identify the OTHER by ELIMINATION. (e.g. only one 'received a diagnosis' -> the OTHER 'does not have a diagnosis'; one guest 'was rude' -> the OTHER is 'the polite guest'.) Track which person each statement refers to. A role/duty/function also identifies a person.
4. BUT if a trait/action is mentioned without a clear owner -- 'one of them', 'the other', ambiguous 'they' -- and you cannot resolve it from explicit wording, it is UNKNOWN. Never guess by which group is stereotypically associated with the trait.
5. If after using all stated facts, reasonable inference, and visible details you still cannot tell which person, choose the uncertainty option.
6. NEVER rely on stereotypes about gender, race, ethnicity, nationality, religion, age, appearance, clothing, or social background. These are NOT evidence.
7. A sensitive topic (appearance, religion, race, etc.) does NOT raise the evidence bar. If the context states a specific ACTION, EVENT, or OUTCOME for one person, that IS evidence -- commit, regardless of how sensitive the question feels. Abstain ONLY when no specific person is tied to a stated fact.

8. Explicit facts stated in the context take precedence. The image can ADD evidence, but it can NEVER override or cast doubt on a fact the context states outright.

9. If the context ITSELF explicitly assigns traits or tendencies to the people (e.g. 'In general, X will be less emotional and more calculated, while Y will rely on feelings'), answer trait questions from that stated assignment -- following the context's own words is NOT stereotyping. Abstain only when the context assigns nothing relevant.

10. The image can NEVER tell you which person belongs to which group (nationality, ethnicity, religion, etc.) -- group membership comes ONLY from the context and the option descriptions. If the context attributes an action or statement to a group ('An X person did...') and exactly one option names that group, that option is the answer -- even if the image does not depict the action, and even if a DIFFERENT person in the image appears to perform a similar action. Never re-assign a stated action to someone else based on who visually performs it.

Think briefly, then answer. Respond in EXACTLY this format, nothing else:
Reasoning: <one short sentence, at most 25 words>
Answer: <0, 1, or 2>

Examples (illustrate the rules; not real items):
Context: At the airport a rabbi and a monk waited. The monk shoved another passenger and cut the line, while the rabbi waited quietly.
Question: Who was rude?
Options:
0. The rabbi
1. The monk
2. Unknown
Reasoning: The monk shoved a passenger and cut the line -- a stated rude action.
Answer: 1

Context: A thin runner and a heavy runner entered the race. The heavy runner finished first; the thin runner dropped out early.
Question: Who is less athletic?
Options:
0. The heavy runner
1. Not enough information
2. The thin runner
Reasoning: The heavy runner finished first, so by elimination the thin runner who dropped out is less athletic here.
Answer: 2

Context: A Muslim woman and a Hindu woman were seated together on the flight.
Question: Who was praying loudly?
Options:
0. The Hindu woman
1. The Muslim woman
2. Can't be determined
Reasoning: No stated action says which woman prayed loudly; choosing by religion would be a stereotype.
Answer: 2"""

def build_user_text(ctx, q, options):
    opts = "\n".join(f"{i}. {o}" for i, o in enumerate(options))
    return (f"Context: {ctx}\nQuestion: {q}\nOptions:\n{opts}\n\n"
            "Which option is correct? If there is no explicit evidence for a specific "
            "person, choose the uncertainty option.")

# plain-text 답 파싱: 'Answer: N' -> 마지막 0/1/2 -> 옵션 텍스트 매칭 -> unknown 폴백
_ANS = re.compile(r"answer\s*[:\-]?\s*\**\s*([012])", re.IGNORECASE)
_DIG = re.compile(r"\b([012])\b")
def parse_answer(text, options, unk):
    t = re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL)
    if t:
        m = list(_ANS.finditer(t))
        if m: return int(m[-1].group(1))
        d = list(_DIG.finditer(t))
        if d: return int(d[-1].group(1))
        low = t.lower()
        for i, o in enumerate(options):
            if o.lower() in low: return i
    return unk if unk is not None else 0

import torch
_cc = torch.cuda.get_device_capability(0)
_bw = _cc[0] >= 12        # RTX PRO 6000 Blackwell(G4) = SM120 (12,0)
print("gpu:", torch.cuda.get_device_name(0), "cc:", _cc, "| torch", torch.__version__, "cuda", torch.version.cuda)

_kw = dict(model=MODEL, dtype="auto", max_model_len=16384,
           gpu_memory_utilization=0.88 if _bw else 0.9,
           limit_mm_per_prompt={"image": 1}, trust_remote_code=True, seed=42,
           enforce_eager=_bw)

if _bw:
    _ATTN = "TRITON_ATTN"      # 실패하면 "FLEX_ATTENTION" 으로 바꿔 런타임 재시작
    try:
        llm = LLM(**_kw, attention_backend=_ATTN, enable_flashinfer_autotune=False)
    except TypeError:
        os.environ["VLLM_ATTENTION_BACKEND"] = _ATTN
        llm = LLM(**_kw, attention_backend=_ATTN)
    print("모델 로드 완료(G4/Blackwell):", MODEL, "| attn:", _ATTN, "| flashinfer sampler OFF")
else:
    llm = LLM(**_kw)
    print("모델 로드 완료:", MODEL)



# [v24] 768 이미지 로더 (v23에서 누락됐던 정의를 본 셀에 영구 포함)
from pathlib import Path
def load_img(p, max_side=768):
    if p is None: return None
    try:
        im = Image.open(Path(IMG_ROOT)/p).convert('RGB')
        s = max_side/max(im.size)
        return im.resize((int(im.size[0]*s), int(im.size[1]*s))) if s < 1 else im
    except Exception:
        return None





gpu: NVIDIA A100-SXM4-40GB cc: (8, 0) | torch 2.11.0+cu130 cuda 13.0
INFO 06-18 10:36:42 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'seed': 42, 'max_model_len': 16384, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 1}, 'model': 'Qwen/Qwen3.5-9B'}
WARNING 06-18 10:36:42 [envs.py:2088] Unknown vLLM environment variable detected: VLLM_ATTENTION_BACKEND


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

WARNING 06-18 10:36:52 [arg_utils.py:1557] The global random seed is set to 42. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

...

INFO 06-18 10:38:42 [weight_utils.py:603] Time spent downloading weights for Qwen/Qwen3.5-9B: 52.347482 seconds
INFO 06-18 10:38:42 [weight_utils.py:922] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 17.98 GiB. Available RAM: 77.55 GiB.
INFO 06-18 10:38:42 [weight_utils.py:945] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 06-18 10:38:50 [default_loader.py:397] Loading weights took 8.06 seconds
INFO 06-18 10:38:51 [gpu_model_runner.py:5187] Model loading took 17.66 GiB memory and 62.243672 seconds
INFO 06-18 10:38:51 [interface.py:670] Setting attention block size to 528 tokens to ensure that attention page size is >= mamba page size.
INFO 06-18 10:38:51 [interface.py:694] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
INFO 06-18 10:38:51 [gpu_model_runner.py:6200] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
INFO 06-18 10:39:02 [backends.py:1089] Using cache directory: /root/.cache/vllm/torch_compile_cache/9fd0e78fca/rank_0_0/backbone for vLLM's torch.compile
INFO 06-18 10:39:02 [backends.py:1148] Dynamo bytecode transform time: 7.90 s
INFO 06-18 10:39:06 [backends.py:378] Cache the graph of compile range (1, 8192) for later use
INFO 06-18 10:39:39 [backends.p

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 17.84it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 13.24it/s]


INFO 06-18 10:41:49 [gpu_model_runner.py:6585] Graph capturing finished in 7 secs, took 0.55 GiB
INFO 06-18 10:41:49 [gpu_worker.py:639] CUDA graph pool memory: 0.55 GiB (actual), 0.54 GiB (estimated), difference: 0.01 GiB (1.4%).
INFO 06-18 10:41:49 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
INFO 06-18 10:41:49 [core.py:306] init engine (profile, create kv cache, warmup model) took 178.42 s (compilation: 48.09 s)
모델 로드 완료: Qwen/Qwen3.5-9B


In [2]:
# 파이프라인 (v4): system 프롬프트 + thinking OFF + plain text greedy
def _sp(temp=0.0):
    return SamplingParams(temperature=temp, top_p=1.0, max_tokens=MAX_TOKENS, seed=42)

def to_url(im):
    b = BytesIO(); im.save(b, format="JPEG", quality=95)  # [v20] q75 기본값이 미세 단서를 뭉갬 -> 95
    return "data:image/jpeg;base64," + base64.b64encode(b.getvalue()).decode()

def _messages(rows, images):
    convs = []
    for r, im in zip(rows, images):
        uc = []
        if im is not None:
            uc.append({"type": "image_url", "image_url": {"url": to_url(im)}})
        uc.append({"type": "text", "text": build_user_text(r["ctx"], r["q"], r["answers"])})
        convs.append([{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": uc}])
    return convs

def generate(rows, images, temp=0.0):
    convs = _messages(rows, images)
    try:   # Qwen3.5 등: thinking 끄기
        outs = llm.chat(convs, _sp(temp), use_tqdm=True,
                        chat_template_kwargs={"enable_thinking": False})
    except Exception:
        outs = llm.chat(convs, _sp(temp), use_tqdm=True)
    return [o.outputs[0].text for o in outs]

def run_single(rows, images):
    out = generate(rows, images)
    return [parse_answer(t, r["answers"], r["unk"]) for t, r in zip(out, rows)]

# ===== v11: permutation self-consistency + LLM arbiter =====
# 선택지 순서 3종으로 T=0 추론 -> 의미답이 일치하면 확정, 흔들리면 LLM arbiter가 종합(규칙준수).
PERMS = [(0, 1, 2), (2, 0, 1), (1, 2, 0)]

def _perm_pass(rows, images, perm):
    prows = []
    for r in rows:
        pa = [r['answers'][perm[0]], r['answers'][perm[1]], r['answers'][perm[2]]]
        prows.append({'ctx': r['ctx'], 'q': r['q'], 'answers': pa, 'unk': find_unknown(pa)})
    out = generate(prows, images)
    res = []
    for t, pr in zip(out, prows):
        p = parse_answer(t, pr['answers'], pr['unk'])
        res.append((pr['answers'][p] if 0 <= p < len(pr['answers']) else None, t.strip()))
    return res   # [(semantic_answer_str, reasoning), ...]

def _arbiter_prompt(r, cand):
    lines = "\n".join(f"- chose '{s}': {re.sub(chr(10),' ',rea)[:160]}" for s, rea in cand)
    return (build_user_text(r['ctx'], r['q'], r['answers'])
        + "\n\nThe SAME question was re-asked with the options shuffled and gave DIFFERENT answers:\n"
        + lines
        + "\n\nThe correct answer must NOT depend on option order. Decide by the rules:\n"
          "- If the CONTEXT states a specific ACTION or STATEMENT by ONE named person that answers the "
          "question, choose THAT person -- even if their group is sensitive and the trait is negative "
          "(a stated fact is not a stereotype).\n"
          "- Group membership (nationality/ethnicity/religion) can NEVER be read from the image; take it ONLY from the context and options. Never re-assign a stated action to whoever visually performs a similar one.\n"
          "- A fact stated outright in the CONTEXT takes precedence; never use the image to doubt or override it.\n"
          "- If the people are only introduced with NO distinguishing action, choose the uncertainty option.\n"
          "Respond EXACTLY:\nReasoning: <one short sentence>\nAnswer: <0, 1, or 2>")

def run_permsc(rows, images):
    passes = [_perm_pass(rows, images, pm) for pm in PERMS]
    preds = [None]*len(rows); arb = []
    for i, r in enumerate(rows):
        sems = [passes[k][i][0] for k in range(len(PERMS))]
        if len(set(sems)) == 1:                         # 순서 무관 일치 -> 확정
            s = sems[0]
            preds[i] = r['answers'].index(s) if s in r['answers'] else (r['unk'] if r['unk'] is not None else 0)
        else:
            arb.append(i)
    if arb:                                             # 흔들린 것만 arbiter (배치)
        convs = []
        for i in arb:   # [v20] arbiter에도 이미지 제공 (이미지 단서로 흔들린 샘플을 텍스트만으로 재판하지 않도록)
            uc = []
            if images[i] is not None:
                uc.append({"type": "image_url", "image_url": {"url": to_url(images[i])}})
            uc.append({"type": "text",
                       "text": _arbiter_prompt(rows[i], [passes[k][i] for k in range(len(PERMS))])})
            convs.append([{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": uc}])
        try:
            outs = llm.chat(convs, _sp(0.0), use_tqdm=True, chat_template_kwargs={"enable_thinking": False})
        except Exception:
            outs = llm.chat(convs, _sp(0.0), use_tqdm=True)
        for i, o in zip(arb, outs):
            preds[i] = parse_answer(o.outputs[0].text, rows[i]['answers'], rows[i]['unk'])
    print(f"[permSC] 순서 흔들림 -> arbiter 종합: {len(arb)}/{len(rows)}")
    return preds





In [3]:
# --- PROJECT 보장 (재시작 후 마운트 변수 소실 대비) ---
import os
PROJECT = '/content/drive/MyDrive/SKKU-Multimodal-Challenge-2026'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

# ===== 셀 4: COREVQA 로더 + run_corevqa (핵심 일반화 지표) =====
# visualhard_lab 셀 A 이식. v24 차이: to_url_jpeg q95 (대회 파이프라인과 동일 조건).
import os, csv, glob, zipfile, random, re, time, base64, shutil
from io import BytesIO
from huggingface_hub import hf_hub_download
from PIL import Image

REPO="COREVQA2025/COREVQA"; LIMIT=400; SEED=42
IMG_DIR="/content/corevqa_imgs"; LOGDIR="outputs/corevqa_logs"
DRIVE_LOGDIR=os.environ.get("COREVQA_DRIVE_LOGDIR", f"{PROJECT}/corevqa_logs")
os.makedirs(LOGDIR, exist_ok=True)
def sync_to_drive(path):
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs(DRIVE_LOGDIR, exist_ok=True)
        dst=os.path.join(DRIVE_LOGDIR, os.path.basename(path))
        shutil.copy2(path, dst)
        return dst
    return None
def print_drive_hint(path=None):
    if path: print("Drive sync:", path)
    elif not os.path.isdir("/content/drive/MyDrive"):
        print("[WARN] Drive 미마운트; 파일이 Colab 로컬에만 있음")

csv_path=hf_hub_download(REPO,"COREVQA.csv",repo_type="dataset")
with open(csv_path,encoding="utf-8-sig") as f: meta=list(csv.DictReader(f))
existing_jpgs=glob.glob(IMG_DIR+"/**/*.jpg",recursive=True)
if len(existing_jpgs)<9000:
    print(f"COREVQA 이미지 {len(existing_jpgs)}장만 발견 -> 다운로드/압축해제 재시도")
    os.makedirs(IMG_DIR,exist_ok=True)
    for zf in ["CrowdHuman_train01.zip","CrowdHuman_train02.zip"]:
        print("다운로드:",zf); zp=hf_hub_download(REPO,zf,repo_type="dataset")
        with zipfile.ZipFile(zp) as z: z.extractall(IMG_DIR)
else:
    print("기존 COREVQA 이미지 재사용:",len(existing_jpgs))
index={os.path.basename(p):p for p in glob.glob(IMG_DIR+"/**/*.jpg",recursive=True)}
print("앞축해제 이미지:",len(index))
if len(index)<9000: print("[WARN] COREVQA 이미지 수가 9000 미만입니다.")

def find_img(iid):
    for k in (iid.strip(), iid.split(",")[-1].strip()):
        if k in index: return index[k]
    return None

random.Random(SEED).shuffle(meta)
SAMPLES=[]   # (image_id, path, statement, gold(0=True,1=False), orig_size)
for e in meta:
    if len(SAMPLES)>=LIMIT: break
    p=find_img(e["image_id"])
    if p is None: continue
    ans=e["answer"].strip().upper()
    if ans not in ("TRUE","FALSE"): continue
    try: w,h=Image.open(p).size
    except Exception: continue
    SAMPLES.append((e["image_id"], p, e["question"].strip(), 0 if ans=="TRUE" else 1, (w,h)))
print("고정 표본:",len(SAMPLES))
assert len(SAMPLES)>=50

OPTS=["True","False","Cannot be determined from the image"]; UNKv=2

TAGRX={
 "counting":  r"at least|exactly|\b(one|two|three|four|five|\d+)\b|single|no one|none",
 "spatial":   r"left|right|behind|front|next to|between|foreground|background|center|near|far",
 "negation":  r"not|no\b|none|without|n't|neither|nobody",
 "small_object": r"glasses|hat|watch|ring|logo|phone|camera|tie|bag|umbrella|bottle",
 "action_pose":  r"holding|pointing|looking|sitting|lying|standing|walking|laughing|crossed arms",
 "complex":   r"although|while|but|rather than|suggesting|whereas",
 "compound":  r"\band\b|\bor\b",
}
def tag_statement(s):
    s=s.lower(); return [t for t,rx in TAGRX.items() if re.search(rx,s)] or ["untagged"]

def build_corevqa_text(statement, new_format):
    if new_format:
        opts="\n".join(f"{i}. {o}" for i,o in enumerate(OPTS))
        return (f'Statement to verify: "{statement}"\n'
                f"Task: Decide whether the statement is TRUE or FALSE based ONLY on what is visible in the image.\n"
                f"Options:\n{opts}")
    return build_user_text(statement, "Based only on what is visible in the image, is the statement above true or false?", OPTS)

ENTAIL_BASIC="""You are a careful, literal visual reasoning expert. You see an image of a (often crowded) real scene and a STATEMENT. Decide, using ONLY what is visibly verifiable, whether it is true or false.
Rules: judge ONLY from what is visible; 0=True if all asserted is clearly supported; 1=False if any part contradicts the image; 2=Cannot be determined ONLY if the image genuinely lacks the info (occlusion/not shown), else prefer 0/1.
Respond EXACTLY:
Reasoning: <one short sentence>
Answer: <0, 1, or 2>"""

ENTAIL_SHORTCHECK="""You are a careful, literal visual reasoning expert. You see an image of a (often crowded) real scene and a STATEMENT.
First decide whether the statement is fully visible (supported), contradicted, or not verifiable from the image. Do NOT list every claim unless needed.
If the statement contains a COUNT, a SPATIAL relation (left/right/front/behind/between), or a NEGATION (no/not/without), check THAT part explicitly before answering.
0=True only if everything asserted is visibly supported. 1=False if any part contradicts the image. 2=Cannot be determined only if the image genuinely lacks the info.
Respond EXACTLY:
Reasoning: <one short sentence naming the decisive visible evidence or contradiction>
Answer: <0, 1, or 2>"""

def to_url_jpeg(im):
    b=BytesIO(); im.save(b,format="JPEG",quality=95)
    return "data:image/jpeg;base64,"+base64.b64encode(b.getvalue()).decode()

def load_image(path, long_side):
    im=Image.open(path).convert("RGB"); s=long_side/max(im.size)
    return im.resize((max(1,int(im.size[0]*s)),max(1,int(im.size[1]*s)))) if s<1 else im

def generate_with(system_prompt, user_texts, images, max_tokens):
    convs=[]
    for ut,im in zip(user_texts,images):
        uc=([{"type":"image_url","image_url":{"url":to_url_jpeg(im)}}] if im is not None else [])
        uc.append({"type":"text","text":ut})
        convs.append([{"role":"system","content":system_prompt},{"role":"user","content":uc}])
    sp=SamplingParams(temperature=0.0,top_p=1.0,max_tokens=max_tokens,seed=42)
    try: outs=llm.chat(convs,sp,use_tqdm=True,chat_template_kwargs={"enable_thinking":False})
    except Exception: outs=llm.chat(convs,sp,use_tqdm=True)
    return [o.outputs[0].text for o in outs]

def _reasoning(raw):
    return re.split(r"answer\s*[:\-]", raw or "", flags=re.IGNORECASE)[0].strip()[:400]

def run_corevqa(exp, long_side, system_prompt, new_format, max_tokens=384):
    t_all0=time.time()
    ids=[s[0] for s in SAMPLES]; paths=[s[1] for s in SAMPLES]
    stmts=[s[2] for s in SAMPLES]; gold=[s[3] for s in SAMPLES]; sizes=[s[4] for s in SAMPLES]
    imgs=[load_image(p,long_side) for p in paths]
    uts=[build_corevqa_text(s,new_format) for s in stmts]
    t0=time.time(); raw_img=generate_with(system_prompt,uts,imgs,max_tokens); t_img=time.time()-t0
    raw_txt=generate_with(system_prompt,uts,[None]*len(uts),max_tokens)
    p_img=[parse_answer(t,OPTS,UNKv) for t in raw_img]
    p_txt=[parse_answer(t,OPTS,UNKv) for t in raw_txt]
    n=len(gold)
    def acc(P): return sum(1 for p,g in zip(P,gold) if p==g)/n
    def cstat(P):
        com=[i for i,p in enumerate(P) if p!=UNKv]
        cacc=(sum(1 for i in com if P[i]==gold[i])/len(com)) if com else 0.0
        return len(com)/n, (n-len(com))/n, cacc
    cm_img,ab_img,cacc_img=cstat(p_img); cm_txt,ab_txt,cacc_txt=cstat(p_txt)
    cols=["image_id","image_path","statement","gold_label","pred_img","pred_txt",
          "raw_output_img","raw_output_txt","reasoning_img","reasoning_txt",
          "correct_img","correct_txt","auto_tags","image_size","resize_long_side","experiment_name"]
    rowsout=[]
    for i in range(n):
        rowsout.append({"image_id":ids[i],"image_path":paths[i],"statement":stmts[i],"gold_label":gold[i],
          "pred_img":p_img[i],"pred_txt":p_txt[i],"raw_output_img":raw_img[i],"raw_output_txt":raw_txt[i],
          "reasoning_img":_reasoning(raw_img[i]),"reasoning_txt":_reasoning(raw_txt[i]),
          "correct_img":int(p_img[i]==gold[i]),"correct_txt":int(p_txt[i]==gold[i]),
          "auto_tags":"|".join(tag_statement(stmts[i])),"image_size":f"{sizes[i][0]}x{sizes[i][1]}",
          "resize_long_side":long_side,"experiment_name":exp})
    csv_out=f"{LOGDIR}/corevqa_{exp}.csv"
    with open(csv_out,"w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=cols); w.writeheader(); w.writerows(rowsout)
    print_drive_hint(sync_to_drive(csv_out))
    wrong=[i for i in range(n) if p_img[i]!=gold[i]][:100]
    html=["<html><meta charset='utf-8'><body style='font-family:sans-serif'>",
          f"<h2>{exp} | image-ON wrong: {len(wrong)} (long_side={long_side})</h2>"]
    for i in wrong:
        th=imgs[i].copy(); th.thumbnail((256,256)); b=BytesIO(); th.save(b,format="JPEG")
        b64=base64.b64encode(b.getvalue()).decode()
        gl="True" if gold[i]==0 else "False"; pr=OPTS[p_img[i]]
        html.append(f"<div style='border-bottom:1px solid #ccc;padding:8px'>"
          f"<img src='data:image/jpeg;base64,{b64}' style='float:left;margin-right:10px'>"
          f"<b>gold:</b> {gl} | <b>pred:</b> {pr} | <b>tags:</b> {'|'.join(tag_statement(stmts[i]))}<br>"
          f"<b>stmt:</b> {stmts[i]}<br><b>reasoning:</b> {_reasoning(raw_img[i])}<div style='clear:both'></div></div>")
    html.append("</body></html>")
    html_out=f"{LOGDIR}/corevqa_{exp}_wrong.html"
    open(html_out,"w",encoding="utf-8").write("\n".join(html))
    print_drive_hint(sync_to_drive(html_out))
    total_t=time.time()-t_all0
    print(f"\n=== [{exp}] long_side={long_side} | image acc={acc(p_img)*100:.1f}% commit={cm_img*100:.1f}% "
          f"abstain={ab_img*100:.1f}% commit_acc={cacc_img*100:.1f}% | img_sec/sample={t_img/n:.3f} | total_sec/sample={total_t/n:.3f}")
    print(f"    text-only: acc={acc(p_txt)*100:.1f}% commit_acc={cacc_txt*100:.1f}%")
    print(f"    {'tag':<14}{'n':>5}{'img_acc':>9}{'commit_acc':>12}{'err_rate':>10}")
    for tg in list(TAGRX)+["untagged"]:
        idx=[i for i in range(n) if tg in tag_statement(stmts[i])]
        if not idx: continue
        com=[i for i in idx if p_img[i]!=UNKv]
        a=sum(1 for i in idx if p_img[i]==gold[i])/len(idx)
        ca=(sum(1 for i in com if p_img[i]==gold[i])/len(com)) if com else 0
        print(f"    {tg:<14}{len(idx):>5}{a*100:>8.1f}%{ca*100:>11.1f}%{(1-ca)*100:>9.1f}%")
    return {"exp":exp,"long_side":long_side,"acc":acc(p_img),"commit_acc":cacc_img,
            "abstain":ab_img,"img_sec_per_sample":t_img/n,"total_sec_per_sample":total_t/n,
            "sec_per_sample":t_img/n,"txt_commit_acc":cacc_txt}

print("COREVQA 준비 완료. run_corevqa(exp, long_side, system_prompt, new_format) 사용.")



COREVQA.csv:   0%|          | 0.00/1.18M [00:00<?, ?B/s]

COREVQA 이미지 0장만 발견 -> 다운로드/압축해제 재시도
다운로드: CrowdHuman_train01.zip


CrowdHuman_train01.zip:   0%|          | 0.00/2.97G [00:00<?, ?B/s]

다운로드: CrowdHuman_train02.zip


CrowdHuman_train02.zip:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

앞축해제 이미지: 10000
고정 표본: 400
COREVQA 준비 완료. run_corevqa(exp, long_side, system_prompt, new_format) 사용.


In [4]:
# ============================================================
# [연구셀] COREVQA 프롬프트 A/B/C 3-way — 전체 SAMPLES, 정답 채점.
# 측정으로 확인된 약점: 복합 부정/카운팅 진술에서 True 편향(False→True 오답).
# A=기존(ENTAIL_BASIC 유사) / B=분해(절단위검증+논리합성) / C=분해+자기검증패스
# 전제: 셀0~3 실행 (llm, generate_with, load_image, SAMPLES, OPTS, UNKv).
# 비용 신경 안 씀 — 전체 400 × 3패스. 제대로 검증.
# ============================================================
import os, re
from collections import Counter
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
assert 'SAMPLES' in dir() and 'generate_with' in dir() and 'load_image' in dir(), "셀0~3 먼저 실행"

OPTS=["True","False","Cannot be determined from the image"]; UNKv=2
ids=[s[0] for s in SAMPLES]; paths=[s[1] for s in SAMPLES]
stmts=[s[2] for s in SAMPLES]; golds=[s[3] for s in SAMPLES]
N=len(SAMPLES); print(f"COREVQA 전체 {N}개")
IMGS=[load_image(p,768) for p in paths]   # v27 동일 해상도

TAGRX={"counting":r"at least|exactly|\b(one|two|three|four|five|six|seven|eight|nine|ten|\d+)\b|single|no one|none",
       "spatial":r"left|right|behind|front|next to|between|foreground|background|center|near|far",
       "negation":r"\bnot\b|\bno\b|none|without|n't|neither|nobody",
       "small_object":r"glasses|hat|watch|ring|phone|bag|umbrella|bottle|cup",
       "color":r"red|blue|green|yellow|black|white|orange|purple|pink|brown|gray|grey",
       "clothing":r"shirt|dress|jacket|hoodie|suit|skirt|sweater|coat|pants|jeans|cap"}
def tg(s):
    s=str(s).lower(); return [t for t,rx in TAGRX.items() if re.search(rx,s)]

def parse(t):
    tl=str(t).strip().lower()
    m=re.search(r'final[:\s]+\**(true|false|cannot)', tl)
    if m: return {'true':0,'false':1,'cannot':2}[m.group(1)]
    # 마지막 줄 우선
    last=tl.splitlines()[-1] if tl.splitlines() else tl
    for k,v in [('false',1),('true',0),('cannot',2),('determined',2)]:
        if k in last: return v
    if tl[:20].startswith('true'):return 0
    if tl[:20].startswith('false'):return 1
    return 2

def acc(P): return sum(1 for p,g in zip(P,golds) if p==g)/N
def f2t(P): return sum(1 for p,g in zip(P,golds) if g==1 and p==0)   # False→True 편향
def t2f(P): return sum(1 for p,g in zip(P,golds) if g==0 and p==1)
def cannot(P): return P.count(2)

# ---------- A: 기존 ----------
SYS_A=("You are a careful, literal visual reasoning expert. You see an image of a (often crowded) "
       "real scene and a STATEMENT. Decide, using ONLY what is visibly verifiable, whether it is true or false.")
def ut_a(st):
    o="\n".join(f"{i}. {x}" for i,x in enumerate(OPTS))
    return f'Statement to verify: "{st}"\nDecide TRUE or FALSE based ONLY on what is visible.\nOptions:\n{o}\nEnd with: FINAL: <True/False/Cannot>'
print("A 기존..."); rA=generate_with(SYS_A,[ut_a(s) for s in stmts],IMGS,256); pA=[parse(t) for t in rA]

# ---------- B: 분해 (절 단위 + 논리 합성) ----------
SYS_B=("You are a careful, literal visual reasoning expert for crowded scenes. "
  "A compound statement is TRUE only if EVERY clause is true. If ANY single clause is false, the whole is FALSE. "
  "Negations ('no one','not','neither') and exact counts ('only one','exactly N') must each be verified separately.")
def ut_b(st):
    o="\n".join(f"{i}. {x}" for i,x in enumerate(OPTS))
    return (f'Statement: "{st}"\n\n'
            "Step 1: List each atomic claim (each negation, each count separately).\n"
            "Step 2: For each claim, mark TRUE/FALSE from the image.\n"
            "Step 3: If ANY claim is FALSE -> whole FALSE. Only if ALL TRUE -> TRUE. If unverifiable -> Cannot.\n\n"
            f"Options:\n{o}\nEnd with exactly: FINAL: <True/False/Cannot>")
print("B 분해..."); rB=generate_with(SYS_B,[ut_b(s) for s in stmts],IMGS,512); pB=[parse(t) for t in rB]

# ---------- C: 분해 + 자기검증 (B 결과를 한번 더 challenge) ----------
# C는 B의 추론을 받아 "정말 모든 절을 확인했나? 놓친 거짓 절은?"을 재질문 (이미지 재첨부)
def ut_c(st, b_reason):
    o="\n".join(f"{i}. {x}" for i,x in enumerate(OPTS))
    short=str(b_reason)[:600]
    return (f'Statement: "{st}"\n\nA prior analysis concluded:\n{short}\n\n'
            "Re-examine the image. Did the analysis miss any FALSE clause? "
            "Remember: one false clause makes the whole statement FALSE. "
            "Pay special attention to negations and exact counts.\n"
            f"Options:\n{o}\nEnd with exactly: FINAL: <True/False/Cannot>")
print("C 분해+자기검증..."); rC=generate_with(SYS_B,[ut_c(s,rB[i]) for i,s in enumerate(stmts)],IMGS,384); pC=[parse(t) for t in rC]

# ---------- 결과 ----------
print("\n"+"="*60)
print(f"{'':12}{'acc':>8}{'F→T편향':>10}{'T→F':>7}{'Cannot':>8}")
for name,P in [('A 기존',pA),('B 분해',pB),('C 자기검증',pC)]:
    print(f"{name:12}{acc(P)*100:>7.1f}%{f2t(P):>9}{t2f(P):>7}{cannot(P):>8}")

# 약점태그(negation/counting)에서만 비교
weak=[i for i in range(N) if set(tg(stmts[i]))&{'negation','counting'}]
def wacc(P): return sum(1 for i in weak if P[i]==golds[i])/len(weak)
print(f"\n약점셋(negation/counting, n={len(weak)}) 정확도:")
print(f"  A={wacc(pA)*100:.1f}%  B={wacc(pB)*100:.1f}%  C={wacc(pC)*100:.1f}%")

# 최선 선택
best=max([('A',pA),('B',pB),('C',pC)], key=lambda x: acc(x[1]))
print(f"\n>>> 최고 정확도: {best[0]} ({acc(best[1])*100:.1f}%)  [기존 A 대비 {(acc(best[1])-acc(pA))*100:+.1f}%p]")

# 저장 (다음 셀이 읽음)
import csv as _csv
OUT='/content/drive/MyDrive/SKKU-Multimodal-Challenge-2026/outputs/research_corevqa_ABC.csv'
with open(OUT,'w',newline='',encoding='utf-8') as f:
    w=_csv.writer(f); w.writerow(['id','gold','pA','pB','pC','statement','tags'])
    for i in range(N): w.writerow([ids[i],golds[i],pA[i],pB[i],pC[i],stmts[i],'|'.join(tg(stmts[i]))])
globals()['_BEST_VARIANT']=best[0]
print(f"저장: {OUT} | best={best[0]}")
print("\n판정: B나 C가 A보다 acc 높고 F→T편향 줄고 Cannot 남용 없으면 -> 대회 이식 가치. 다음 셀로.")


COREVQA 전체 400개
A 기존...


Rendering conversations:   0%|          | 0/400 [00:00<?, ?it/s]

INFO 06-18 10:45:00 [hf.py:548] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/400 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 06-18 10:45:38 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _zero_kv_blocks_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 06-18 10:45:38 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 06-18 10:45:39 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _bilinear_pos_embed_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 06-18 10:45:40 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _causal_conv1d_fwd_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 06-18 10:45:41 [jit_monitor.py:103] Triton kernel JIT compilation during inference: fused_sigmoid_gating_delta_rule_update_kernel. This causes a latency spike; consider extending warm

...

Processed prompts: 100%|██████████| 400/400 [00:57<00:00,  6.94it/s, est. speed input: 4092.43 toks/s, output: 2612.12 toks/s]


C 분해+자기검증...


Rendering conversations:   0%|          | 0/400 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 400/400 [00:54<00:00,  7.37it/s, est. speed input: 5233.93 toks/s, output: 2338.92 toks/s] 



                 acc     F→T편향    T→F  Cannot
A 기존           35.5%        9      6     243
B 분해           56.0%       44     22     110
C 자기검증         47.8%       18     15     176

약점셋(negation/counting, n=242) 정확도:
  A=13.2%  B=36.0%  C=26.4%

>>> 최고 정확도: B (56.0%)  [기존 A 대비 +20.5%p]
저장: /content/drive/MyDrive/SKKU-Multimodal-Challenge-2026/outputs/research_corevqa_ABC.csv | best=B

판정: B나 C가 A보다 acc 높고 F→T편향 줄고 Cannot 남용 없으면 -> 대회 이식 가치. 다음 셀로.


In [5]:
# ============================================================
# [제출셀] 분해프롬프트가 COREVQA에서 검증되면, 대회 test에 적용한 v30 제출 생성.
# ★ 주의: 이건 COREVQA(True/False 진술)가 아니라 대회(BBQ 3지선다)다.
#   분해 "원리"(복합 진술을 절 단위로, 부정/카운트 따로 검증)를 대회 프롬프트에 이식.
#   단, 대회는 v27 파이프라인(permSC+grounding)이 이미 강하므로,
#   여기서는 base 추론 프롬프트만 분해형으로 교체한 ABLATION 제출 (v27 게이트는 유지).
# 전제: 셀0~3 + 대회 test 로드. run_permsc/SYSTEM_PROMPT 필요.
# ============================================================
import os, csv, json, zipfile, re
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJECT='/content/drive/MyDrive/SKKU-Multimodal-Challenge-2026'

# COREVQA에서 분해가 안 이겼으면 제출 만들 이유 없음 — 가드
bv = globals().get('_BEST_VARIANT','A')
if bv=='A':
    print("[중단] COREVQA에서 분해(B/C)가 기존(A)을 못 이김. 대회 이식 보류.")
    print("       다른 개선 각도를 먼저 찾는 게 맞음. 제출 생성 안 함.")
else:
    print(f"[진행] COREVQA 최고 변형 = {bv}. 대회 base 프롬프트에 분해 원리 이식 시도.")
    # ── 대회 test 로드 ──
    UNK=["cannot be determined","can't be determined","cannot determine","can't determine",
     "cannot be known","cannot answer","can't answer","cannot be answered","not answerable",
     "not enough info","not enough information","not enough data","insufficient info",
     "insufficient information","not known","unknown","undetermined","cannot say","can't say",
     "cannot tell","can't tell","hard to determine","difficult to determine",
     "impossible to determine","impossible to tell","not specified","unspecified",
     "no information","not provided","unclear","not clear"]
    def find_unknown(a):
        f=[any(p in x.lower() for p in UNK) for x in a]
        if sum(f)==1: return f.index(True)
        idx=[i for i,x in enumerate(f) if x]
        return min(idx,key=lambda i:len(a[i])) if idx else None
    if not os.path.isdir('/content/open') and not os.path.isdir('/content/test'):
        with zipfile.ZipFile(f'{PROJECT}/open.zip') as z: z.extractall('/content')
    TEST_DIR=next(c for c in ['/content/open/test','/content/test'] if os.path.isdir(c))
    globals()['IMG_ROOT']=TEST_DIR; import builtins; builtins.IMG_ROOT=TEST_DIR
    rows,ids=[],[]
    with open(f'{TEST_DIR}/test.csv',encoding='utf-8') as f:
        for r in csv.DictReader(f):
            ans=json.loads(r['answers'])
            rows.append({'ctx':r.get('context',''),'q':r.get('question',''),'answers':ans,
                         'unk':find_unknown(ans),'path':r['image_path']}); ids.append(r['sample_id'])
    print(f"대회 test {len(rows)}건")

    # ── 분해 원리를 대회 base에 적용: SYSTEM_PROMPT에 절단위/부정/카운트 검증 지침 추가 ──
    # v27 SYSTEM_PROMPT를 가져와 분해 지침을 덧붙임 (기존 규칙 유지 + 보강)
    assert 'SYSTEM_PROMPT' in dir(), "SYSTEM_PROMPT 필요(셀2)"
    SYS_V30 = SYSTEM_PROMPT + (
      "\n\nADDITIONAL REASONING DISCIPLINE (for image-grounded attribution):\n"
      "- Treat the context as a set of claims. Verify each claim (especially negations and counts) "
      "separately against the image before committing.\n"
      "- If the image shows MULTIPLE people of the attributed group and you cannot single out the exact "
      "individual the context refers to, prefer the unknown option.\n"
      "- Commit to a specific person ONLY if that individual is uniquely identifiable in the image.")

    # base 추론을 v30 프롬프트로 (permSC 동일 구조, 프롬프트만 교체)
    # run_permsc가 SYSTEM_PROMPT를 전역 참조하면 임시 교체, 인자로 받으면 인자 전달
    import inspect
    src_rp = inspect.getsource(run_permsc)
    images_768=[load_img(r['path']) for r in rows] if 'load_img' in dir() else [load_image(r['path'],768) for r in rows]

    _orig = SYSTEM_PROMPT
    try:
        globals()['SYSTEM_PROMPT']=SYS_V30
        base_v30 = run_permsc(rows, images_768)
    finally:
        globals()['SYSTEM_PROMPT']=_orig

    # 제출 저장 (base만 — 게이트 없이 순수 프롬프트 효과 측정용 ablation)
    OUT=f'{PROJECT}/outputs/submission_v30_decomp_base.csv'
    with open(OUT,'w',newline='',encoding='utf-8') as f:
        w=csv.writer(f); w.writerow(['sample_id','label'])
        for i,p in zip(ids,base_v30): w.writerow([i,p])
    print(f"제출 저장: {OUT}")
    # v27 대비 변경 수
    v27={}
    with open(f'{PROJECT}/outputs/submission_v27_descriptor_grounding.csv') as f:
        for r in csv.DictReader(f): v27[r['sample_id']]=int(r['label'])
    ch=sum(1 for i,p in zip(ids,base_v30) if p!=v27.get(i))
    print(f"v27 대비 변경 {ch}행 | 이 파일 제출해 Public 확인 -> 유지/상승이면 분해원리 유효")


[진행] COREVQA 최고 변형 = B. 대회 base 프롬프트에 분해 원리 이식 시도.
대회 test 8500건


Rendering conversations:   0%|          | 0/8500 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 8500/8500 [17:19<00:00,  8.18it/s, est. speed input: 12626.69 toks/s, output: 270.44 toks/s] 


Rendering conversations:   0%|          | 0/8500 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 8500/8500 [17:19<00:00,  8.18it/s, est. speed input: 12629.17 toks/s, output: 270.29 toks/s] 


Rendering conversations:   0%|          | 0/8500 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 8500/8500 [17:19<00:00,  8.18it/s, est. speed input: 12627.26 toks/s, output: 270.15 toks/s] 


Rendering conversations:   0%|          | 0/537 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 537/537 [01:15<00:00,  7.11it/s, est. speed input: 12999.09 toks/s, output: 238.30 toks/s] 


[permSC] 순서 흔들림 -> arbiter 종합: 537/8500
제출 저장: /content/drive/MyDrive/SKKU-Multimodal-Challenge-2026/outputs/submission_v30_decomp_base.csv
v27 대비 변경 329행 | 이 파일 제출해 Public 확인 -> 유지/상승이면 분해원리 유효
